[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/21_gradient_clipping_interview.ipynb)

# 🟢 Solution: Gradient Norm Clipping（面试版）

## 📋 题目描述

**难度：** Easy

实现梯度范数裁剪。

梯度裁剪在所有参数梯度的 L2 范数超过阈值时进行缩放，防止梯度爆炸。

**签名:** `clip_grad_norm(parameters, max_norm) -> float`

**参数:**
- `parameters` — 带 `.grad` 属性的张量列表
- `max_norm` — 允许的最大梯度范数

**返回:** 原始总梯度范数（浮点数）

**约束:**
- 总范数 = `sqrt(sum(p.grad.norm()^2))`
- 仅在总范数 > max_norm 时裁剪
- 保持梯度方向不变

**提示：** 1. total_norm = sqrt(Σ p.grad.norm()²)
2. clip_coef = max_norm / total_norm
3. 若 total_norm > max_norm：所有梯度乘以 clip_coef
4. 返回原始 total_norm（float）


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

def clip_grad_norm(parameters, max_norm):
    # ---- Step 1: 过滤无梯度参数 ----
    parameters = [p for p in parameters if p.grad is not None]

    # ---- Step 2: 计算全局梯度范数 ----
    # 面试要点：为什么是"全局"范数而不是逐参数裁剪？
    # 答：逐参数裁剪会改变不同参数之间的梯度比例
    #     全局裁剪保持所有参数梯度的相对大小不变
    #
    # 数学：把所有参数梯度视为一个长向量 g = [g1, g2, ...]
    #       total_norm = ||g||₂ = sqrt(Σ ||gi||²)
    total_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))

    # ---- Step 3: 裁剪 ----
    # clip_coef = max_norm / total_norm
    # 当 total_norm > max_norm 时，clip_coef < 1
    # 所有梯度乘以同一个系数，等价于将梯度向量缩放到范数 = max_norm
    #
    # 为什么保持方向？因为所有分量等比例缩放
    # 梯度下降方向 = -∇L，缩放后 = -clip_coef * ∇L，方向不变
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in parameters:
            p.grad.mul_(clip_coef)

    return total_norm.item()

# 面试常见追问：
# Q: 梯度裁剪 vs 梯度缩放？
# A: 裁剪是有条件的（只在范数过大时），缩放是无条件的（每次都乘系数）
# Q: 为什么用 L2 范数而不是 L1？
# A: L2 范数数学性质好（可微），且与 SGD 的更新方向自然对齐
# Q: 裁剪后梯度的范数是多少？
# A: 恰好等于 max_norm（如果原来超过的话）


In [ ]:
p1 = torch.randn(3, requires_grad=True)
p2 = torch.randn(3, requires_grad=True)
(p1.sum() + p2.sum()).backward()
print('Before:', clip_grad_norm([p1, p2], max_norm=1.0))
print('p1 grad:', p1.grad)


In [ ]:
from torch_judge import check
check('gradient_clipping')


## 📝 核心思路总结

1. **全局范数**：所有参数梯度视为一个向量，计算其 L2 范数
2. **条件裁剪**：只在范数超过 max_norm 时缩放，否则不动
3. **方向不变**：所有梯度等比例缩放，保持下降方向
4. **数值安全**：加 epsilon 防止除以零
